# 2. Inverse probability of treatment weighting (IPTW): RHC data

Weight each patient by 1 / P(the treatment they actually got | covariates). The weighted population has no
association between covariates and treatment, so a marginal structural model (MSM) fit to it estimates the
causal effect of RHC on death.

In [ ]:
source("setup.R")
use_packages("tableone", "ipw", "sandwich", "survey")  # sandwich: robust variance estimation

mydata <- load_rhc()
xvars <- c("age", "female", "meanbp1", "ARF", "CHF", "Cirr", "colcan", "Coma", "lungcan", "MOSF", "sepsis")

In [ ]:
# Table 1 before weighting
table1 <- CreateTableOne(vars = xvars, strata = "treatment", data = mydata, test = FALSE)
print(table1, smd = TRUE)

# propensity score model
psmodel <- glm(treatment ~ age + female + meanbp1 + ARF + CHF + Cirr + colcan + Coma + lungcan + MOSF + sepsis,
               family = binomial(link = "logit"), data = mydata)
ps <- predict(psmodel, type = "response")

# weights: 1/PS for the treated, 1/(1-PS) for controls
weight <- ifelse(mydata$treatment == 1, 1 / ps, 1 / (1 - ps))

# weighted Table 1
weighteddata <- svydesign(ids = ~1, data = mydata, weights = ~weight)
weightedtable <- svyCreateTableOne(vars = xvars, strata = "treatment", data = weighteddata, test = FALSE)
print(weightedtable, smd = TRUE)

In [ ]:
# weighted mean age of the treated, by brute force (matches the weighted table)
with(mydata, sum(weight[treatment == 1] * age[treatment == 1]) / sum(weight[treatment == 1]))

## Marginal structural models

Weighted GLMs of `died` on `treatment`, with sandwich (robust) standard errors to account for the weighting.
- log link: $\log E(Y^a) = \psi_0 + \psi_1 a$, so $e^{\psi_1}$ is the causal relative risk
- identity link: $E(Y^a) = \psi_0 + \psi_1 a$, so $\psi_1$ is the causal risk difference

In [ ]:
# estimate and 95% CI for the treatment coefficient of a weighted GLM, transformed by `f`
msm_effect <- function(link, weights, f = identity) {
  fit <- glm(died ~ treatment, weights = weights, family = quasibinomial(link = link), data = mydata)
  beta <- unname(coef(fit)[2])
  se <- unname(sqrt(diag(vcovHC(fit, type = "HC0")))[2])
  f(c(lower = beta - 1.96 * se, estimate = beta, upper = beta + 1.96 * se))
}

print("causal relative risk")
msm_effect("log", weight, exp)

print("causal risk difference")
msm_effect("identity", weight)

print("causal risk difference, weights truncated at 10")
msm_effect("identity", pmin(weight, 10))

## The same with the `ipw` package

`ipwpoint` fits the propensity score model and computes the weights; `svyglm` fits the MSM (risk difference).
The second fit truncates the weights at their 1st and 99th percentiles.

In [ ]:
weightmodel <- ipwpoint(exposure = treatment, family = "binomial", link = "logit",
                        denominator = ~ age + female + meanbp1 + ARF + CHF + Cirr + colcan + Coma + lungcan + MOSF + sepsis,
                        data = mydata)
summary(weightmodel$ipw.weights)
ipwplot(weights = weightmodel$ipw.weights, logscale = FALSE, main = "weights", xlim = c(0, 22))

mydata$wt <- weightmodel$ipw.weights
msm <- svyglm(died ~ treatment, design = svydesign(~1, weights = ~wt, data = mydata))
coef(msm)
confint(msm)

In [ ]:
weightmodel <- ipwpoint(exposure = treatment, family = "binomial", link = "logit",
                        denominator = ~ age + female + meanbp1 + ARF + CHF + Cirr + colcan + Coma + lungcan + MOSF + sepsis,
                        data = mydata, trunc = .01)
summary(weightmodel$weights.trun)
ipwplot(weights = weightmodel$weights.trun, logscale = FALSE, main = "truncated weights", xlim = c(0, 22))

mydata$wt <- weightmodel$weights.trun
msm <- svyglm(died ~ treatment, design = svydesign(~1, weights = ~wt, data = mydata))
coef(msm)
confint(msm)